![image.png](https://i.imgur.com/4fN73lZ.png)

# Soft Actor-Critic (SAC) from scratch

We implement **SAC** (Haarnoja et al., 2018) from scratch in PyTorch and train it on
**Pendulum-v1**, then reuse the exact same code on the **Walker2d** MuJoCo task for a
direct comparison with Day 5's DDPG/TD3.

SAC is the **maximum-entropy** off-policy actor-critic. Compared with Day 5's DDPG/TD3
it changes three things, all motivated in the lecture:

- a **stochastic, squashed-Gaussian actor** $\pi_\phi$ instead of a deterministic $\mu_\theta$;
- an **entropy bonus** $\alpha\,\mathcal{H}(\pi)$ in the objective, so exploration is *built in* — no hand-tuned action noise;
- **clipped double-Q** soft critics (inherited from TD3), and finally **automatic temperature** tuning.

$$J(\pi) = \sum_t \mathbb{E}_{(s_t,a_t)\sim\rho_\pi}\big[\, r(s_t,a_t) + \alpha\,\mathcal{H}(\pi(\cdot\mid s_t)) \,\big]$$

> **Exercise version.** Three `# TASK`s are left for you — each one is a piece that is
> *new in SAC*, not a repeat of earlier days (the clipped double-Q `min` and the
> critic-climb actor loss, both from Day 5, are given). A complete reference is in
> `Day-6_SAC_Custom_Pytorch_Pendulum_Solution.ipynb`.
> - **TASK 1** — the tanh-Jacobian log-prob correction (squashed Gaussian actor).
> - **TASK 2** — the entropy-augmented *soft* target (the max-entropy core).
> - **TASK 3** — the automatic temperature loss (in the SAC v2 extension).

## Setup

We install with **uv** and render rollouts as GIFs with `imageio` (consistent with the
other RL labs). Pendulum needs `gymnasium[classic-control]`; the optional Walker2d
comparison at the end needs `gymnasium[mujoco]`.

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" "gymnasium[classic-control,mujoco]" imageio matplotlib torch

# Content

**Pendulum-v1** is the classic 1-D continuous-control sanity task: swing a pendulum
upright and hold it there. The single action is a torque in $[-2, 2]$; the reward is
negative (angle + effort penalty), so a good policy pushes the return from about
$-1500$ (random) up toward $\gtrsim -200$. You can read about its observation, action
and reward [here](https://gymnasium.farama.org/environments/classic_control/pendulum/).

![Pendulum](https://gymnasium.farama.org/_images/pendulum.gif)

In [ ]:
import collections
import os
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Tiny MLPs + a cheap env step are CPU-bound, so CPU is typically as fast as GPU here.
device = torch.device("cpu")
print("device:", device)

## Replay buffer

Being **off-policy**, SAC reuses past transitions from a replay buffer $\mathcal{D}$ —
the same buffer we used for DDPG on Day 5, unchanged.

In [ ]:
Experience = collections.namedtuple(
    "Experience", ["state", "action", "reward", "next_state", "done"])


class ExperienceBuffer:
    """A fixed-size replay buffer of past transitions."""
    def __init__(self, capacity):
        self.buffer = collections.deque(maxlen=capacity)

    def __len__(self):
        return len(self.buffer)

    def append(self, experience):
        self.buffer.append(experience)

    def sample(self, batch_size):
        idx = np.random.choice(len(self.buffer), batch_size, replace=False)
        states, actions, rewards, next_states, dones = zip(*[self.buffer[i] for i in idx])
        t = lambda x: torch.tensor(np.array(x, dtype=np.float32), device=device)
        return (t(states), t(actions), t(rewards).unsqueeze(1),
                t(next_states), t(dones).unsqueeze(1))

## The squashed-Gaussian actor

Instead of DDPG's deterministic $\mu_\theta(s)$, SAC's actor outputs a **Gaussian**
$\mathcal{N}(\mu_\phi(s), \sigma_\phi(s))$ over actions. To sample and still get gradients
we use the **reparameterization trick**: $u = \mu_\phi + \sigma_\phi \odot \epsilon$ with
$\epsilon\sim\mathcal{N}(0, I)$, then squash with $\tanh$ to bound the action:
$$a = \tanh(u).$$

The squash means the density is no longer a plain Gaussian — we must correct
$\log\pi$ by the **log-determinant of the $\tanh$ Jacobian**:
$$\log\pi(a\mid s) = \log\mathcal{N}(u\mid \mu_\phi,\sigma_\phi) - \sum_i \log\!\big(1 - \tanh^2(u_i)\big).$$

This correction is unique to the squashed policy — there was no analogue in the Gaussian
policies of Days 3–4. At evaluation we drop the noise and act with the mean, $a = \tanh(\mu_\phi)$.

In [ ]:
LOG_STD_MIN, LOG_STD_MAX = -20, 2   # keep sigma in a sane range


class SquashedGaussianActor(nn.Module):
    """Stochastic policy: state -> tanh-squashed Gaussian action, with log-prob."""
    def __init__(self, state_dim, action_dim, action_scale):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, 256)
        self.fc2 = nn.Linear(256, 256)
        self.mu_head = nn.Linear(256, action_dim)
        self.log_std_head = nn.Linear(256, action_dim)
        self.action_scale = action_scale

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        mu = self.mu_head(x)
        log_std = self.log_std_head(x).clamp(LOG_STD_MIN, LOG_STD_MAX)
        return mu, log_std

    def sample(self, state):
        """Reparameterized action sample and its log-probability."""
        mu, log_std = self.forward(state)
        std = log_std.exp()
        normal = torch.distributions.Normal(mu, std)
        u = normal.rsample()                 # reparameterized: u = mu + std * eps
        a = torch.tanh(u)                    # squash into (-1, 1)
        # TASK 1: log-prob of the squashed action, WITH the tanh change-of-variables correction.
        #   A plain Gaussian log-prob is wrong here because tanh compresses the density;
        #   subtract the log-det of the tanh Jacobian, then sum over action dims.  This
        #   squashing correction is unique to SAC's bounded actor (no analogue in Days 3-4).
        # HINT: log_prob = normal.log_prob(u) - log(1 - tanh(u)^2 + 1e-6)
        #       then reduce with .sum(dim=1, keepdim=True) -> shape (B, 1)
        log_prob = None
        if log_prob is None:
            raise NotImplementedError("TASK 1: tanh Jacobian log-prob correction")
        return a * self.action_scale, log_prob

    def mean_action(self, state):
        """Deterministic action for evaluation: tanh(mu)."""
        mu, _ = self.forward(state)
        return torch.tanh(mu) * self.action_scale

## Twin soft critics

Like TD3 (Day 5) we learn **two** critics $Q_{\theta_1}, Q_{\theta_2}$ and use the
**minimum** of their targets — this "clipped double-Q" trick fights the overestimation
bias. Each is an ordinary $Q(s,a)$ network; nothing SAC-specific here — you built exactly
this on Day 5.

In [ ]:
class Critic(nn.Module):
    """Soft action-value function Q(s, a)."""
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc1 = nn.Linear(state_dim + action_dim, 256)
        self.fc2 = nn.Linear(256, 256)
        self.fc3 = nn.Linear(256, 1)

    def forward(self, state, action):
        x = F.relu(self.fc1(torch.cat([state, action], dim=1)))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

## The SAC agent (v1, fixed temperature)

Putting it together. The one genuinely new idea over Day 5 lives in the critic target:
the bootstrap uses a **soft value** — the clipped double-Q value *minus an entropy bonus*:
$$V_{\text{soft}}(s') = \min_{i=1,2} Q_{\bar\theta_i}(s', \tilde a') - \alpha\log\pi_\phi(\tilde a'\mid s'),\qquad
  y = r + \gamma\,(1-d)\,V_{\text{soft}}(s'),\quad \tilde a'\sim\pi_\phi(\cdot\mid s').$$

The actor then minimizes the KL to $\exp(Q/\alpha)$, which after reparameterization is a
critic-climb *with* an entropy term (structurally the Day-5 deterministic-policy-gradient
idea, so it is given below):
$$J_\pi(\phi) = \mathbb{E}\big[\alpha\log\pi_\phi(\tilde a\mid s) - \min_{i=1,2} Q_{\theta_i}(s, \tilde a)\big].$$

Here $\alpha$ is a **fixed** hyperparameter (SAC v1); the extension below learns it.

In [ ]:
class SACAgent:
    def __init__(self, env, buffer, args):
        self.env = env
        self.buffer = buffer
        self.args = args

        s_dim = env.observation_space.shape[0]
        a_dim = env.action_space.shape[0]
        self.action_scale = float(env.action_space.high[0])
        self.gamma = args["gamma"]
        self.tau = args["tau"]
        self.alpha = args["alpha"]

        self.actor = SquashedGaussianActor(s_dim, a_dim, self.action_scale).to(device)
        self.q1 = Critic(s_dim, a_dim).to(device)
        self.q2 = Critic(s_dim, a_dim).to(device)
        self.q1_target = Critic(s_dim, a_dim).to(device)
        self.q2_target = Critic(s_dim, a_dim).to(device)
        self.q1_target.load_state_dict(self.q1.state_dict())
        self.q2_target.load_state_dict(self.q2.state_dict())

        self.actor_opt = optim.Adam(self.actor.parameters(), lr=args["actor_lr"])
        self.critic_opt = optim.Adam(
            list(self.q1.parameters()) + list(self.q2.parameters()), lr=args["critic_lr"])

    def select_action(self, state):
        """Stochastic action for data collection — entropy IS the exploration."""
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            a, _ = self.actor.sample(s)
        return a.cpu().numpy()[0]

    def eval_action(self, state):
        """Deterministic mean action for evaluation."""
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            a = self.actor.mean_action(s)
        return a.cpu().numpy()[0]

    def soft_update(self, net, target_net):
        for p, tp in zip(net.parameters(), target_net.parameters()):
            tp.data.copy_(self.tau * p.data + (1 - self.tau) * tp.data)

    def update(self):
        """One SAC gradient step on a sampled mini-batch."""
        states, actions, rewards, next_states, dones = self.buffer.sample(self.args["batch_size"])

        # --- soft critic target ---
        with torch.no_grad():
            next_a, next_logp = self.actor.sample(next_states)
            # clipped double-Q — the same min-of-two-critics trick as TD3 (Day 5)
            min_q = torch.min(self.q1_target(next_states, next_a),
                              self.q2_target(next_states, next_a))
            # TASK 2: the SOFT target — the heart of max-entropy RL.
            #   The clipped double-Q value `min_q` is given (you built this on Day 5).
            #   Turn it into a SOFT value by subtracting the entropy bonus alpha*log pi,
            #   then bootstrap as usual. This entropy term is what makes SAC "soft".
            # HINT: soft_next_value = min_q - alpha * next_logp
            #       y = r + gamma * (1 - done) * soft_next_value
            soft_next_value = None
            y = None
            if y is None:
                raise NotImplementedError("TASK 2: entropy-augmented soft target")
        critic_loss = F.mse_loss(self.q1(states, actions), y) + \
                      F.mse_loss(self.q2(states, actions), y)
        self.critic_opt.zero_grad()
        critic_loss.backward()
        self.critic_opt.step()

        # --- actor loss (given): entropy-regularized critic climb, Day-5 style ---
        a_pi, logp = self.actor.sample(states)
        q_pi = torch.min(self.q1(states, a_pi), self.q2(states, a_pi))
        actor_loss = (self.alpha * logp - q_pi).mean()
        self.actor_opt.zero_grad()
        actor_loss.backward()
        self.actor_opt.step()

        # --- Polyak-average the target critics ---
        self.soft_update(self.q1, self.q1_target)
        self.soft_update(self.q2, self.q2_target)

## Initialize the environment and agent

In [ ]:
env = gym.make("Pendulum-v1")

args = {
    "replay_size": 1_000_000,
    "batch_size": 256,
    "actor_lr": 3e-4,
    "critic_lr": 3e-4,
    "tau": 0.005,
    "gamma": 0.99,
    "alpha": 0.2,            # fixed entropy temperature (SAC v1)
    "start_steps": 1_000,   # random warm-up before learning starts
}

total_steps = 20_000        # Pendulum solves quickly; ~15k is usually enough

buffer = ExperienceBuffer(args["replay_size"])
agent = SACAgent(env, buffer, args)

## Training

One gradient step per environment step. Two things to notice:

- **No injected exploration noise.** Unlike DDPG (Day 5), the action is a *stochastic
  sample* from the policy — the entropy bonus is what drives exploration.
- **Random warm-up.** For the first `start_steps` we act randomly to fill the buffer
  before any learning (a standard off-policy implementation detail).

We bootstrap on time-limit truncation but **not** on true termination, so the done mask
uses `terminated` only.

In [ ]:
def train(agent, env, total_steps, args):
    scores, episode_return = [], 0.0
    state, _ = env.reset(seed=0)
    for step in range(1, total_steps + 1):
        if step < args["start_steps"]:
            action = env.action_space.sample()          # random warm-up
        else:
            action = agent.select_action(state)         # stochastic — entropy is the exploration

        next_state, reward, terminated, truncated, _ = env.step(action)
        agent.buffer.append(Experience(state, action, reward, next_state, float(terminated)))
        state = next_state
        episode_return += reward

        if terminated or truncated:
            scores.append(episode_return)
            episode_return = 0.0
            state, _ = env.reset()

        if len(agent.buffer) >= args["start_steps"]:
            agent.update()

        if step % 2000 == 0 and scores:
            print(f"step {step:6d} | last-10 episode return: {np.mean(scores[-10:]):8.1f}")
    env.close()
    return scores

scores = train(agent, env, total_steps, args)

## Reward curve

In [ ]:
def plot_scores(scores, title, window=10):
    smoothed = np.convolve(scores, np.ones(window) / window, mode="valid")
    plt.figure(figsize=(9, 4))
    plt.plot(smoothed)
    plt.xlabel("episode")
    plt.ylabel(f"episode return ({window}-episode moving avg)")
    plt.title(title)
    plt.grid(alpha=0.3)
    plt.show()

plot_scores(scores, "SAC (v1) on Pendulum")

## Visualizing the trained agent

A small helper rolls out the deterministic (mean-action) policy and saves a looping GIF.
We reuse it for every trained agent below (v1, v2, and Walker2d).

In [ ]:
import imageio.v2 as imageio
from IPython.display import Image, display

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")   # headless pygame (classic-control)
os.environ.setdefault("MUJOCO_GL", "egl")           # headless MuJoCo
os.makedirs("video", exist_ok=True)


def render_gif(env_id, agent, path, steps=300, seed=0):
    """Roll out the deterministic (mean-action) policy and save a looping GIF."""
    renv = gym.make(env_id, render_mode="rgb_array")
    state = renv.reset(seed=seed)[0]
    frames = []
    for _ in range(steps):
        frames.append(renv.render())
        state, _, terminated, truncated, _ = renv.step(agent.eval_action(state))
        if terminated or truncated:
            break
    renv.close()
    imageio.mimsave(path, frames, fps=30, loop=0)   # loop=0 -> replays forever
    return Image(filename=path)


display(render_gif("Pendulum-v1", agent, "video/sac_v1_pendulum.gif", steps=200))

## Extension — Automatic temperature tuning (SAC v2)

SAC v1 leaves one brittle knob: the temperature $\alpha$. Too high and the policy stays
random; too low and it collapses to greedy. **SAC v2** removes it by turning "explore
enough" into a **constraint** — keep the policy's entropy above a target $\bar{\mathcal{H}}$
— and solving the dual. The Lagrange multiplier *is* $\alpha$, and it follows one SGD
step of its own:

$$J(\alpha) = \mathbb{E}_{a\sim\pi_\phi}\big[-\alpha\,(\log\pi_\phi(a\mid s) + \bar{\mathcal{H}})\big],\qquad \bar{\mathcal{H}} = -\dim(\mathcal{A}).$$

Intuition: if the policy is *more* random than the target ($-\log\pi > \bar{\mathcal H}$),
$\alpha$ is pushed **down**; if it is too deterministic, $\alpha$ is pushed **up**. We
optimize $\log\alpha$ so $\alpha$ stays positive. This dual/Lagrangian update has no
analogue in any earlier lab — it is the piece that makes SAC "tuning-free".

In [ ]:
class SACAutoAlpha(SACAgent):
    def __init__(self, env, buffer, args):
        super().__init__(env, buffer, args)
        a_dim = env.action_space.shape[0]
        self.target_entropy = -float(a_dim)                     # Hbar = -dim(A)
        self.log_alpha = torch.zeros(1, requires_grad=True, device=device)
        self.alpha = self.log_alpha.exp().item()
        self.alpha_opt = optim.Adam([self.log_alpha], lr=args["alpha_lr"])

    def update(self):
        states, actions, rewards, next_states, dones = self.buffer.sample(self.args["batch_size"])

        # --- soft critic target (same soft target as v1; alpha is now learned) ---
        with torch.no_grad():
            next_a, next_logp = self.actor.sample(next_states)
            min_q = torch.min(self.q1_target(next_states, next_a),
                              self.q2_target(next_states, next_a))
            soft_next_value = min_q - self.alpha * next_logp
            y = rewards + self.gamma * (1 - dones) * soft_next_value
        critic_loss = F.mse_loss(self.q1(states, actions), y) + \
                      F.mse_loss(self.q2(states, actions), y)
        self.critic_opt.zero_grad()
        critic_loss.backward()
        self.critic_opt.step()

        # --- actor loss (given) ---
        a_pi, logp = self.actor.sample(states)
        q_pi = torch.min(self.q1(states, a_pi), self.q2(states, a_pi))
        actor_loss = (self.alpha * logp - q_pi).mean()
        self.actor_opt.zero_grad()
        actor_loss.backward()
        self.actor_opt.step()

        # TASK 3: automatic temperature loss — the SAC v2 headline, unlike anything earlier.
        #   Keeping entropy >= Hbar is a constraint; its Lagrange multiplier is alpha.
        #   Optimize log_alpha (so alpha>0). Detach log pi: tuning alpha must not move the actor.
        # HINT: alpha_loss = -(log_alpha * (log pi + target_entropy).detach()).mean()
        alpha_loss = None
        if alpha_loss is None:
            raise NotImplementedError("TASK 3: automatic temperature (log-alpha) loss")
        self.alpha_opt.zero_grad()
        alpha_loss.backward()
        self.alpha_opt.step()
        self.alpha = self.log_alpha.exp().item()

        self.soft_update(self.q1, self.q1_target)
        self.soft_update(self.q2, self.q2_target)

In [ ]:
auto_args = dict(args, alpha_lr=3e-4)   # alpha is now learned; its initial value doesn't matter

env = gym.make("Pendulum-v1")
auto_buffer = ExperienceBuffer(auto_args["replay_size"])
auto_agent = SACAutoAlpha(env, auto_buffer, auto_args)

auto_scores = train(auto_agent, env, total_steps, auto_args)
plot_scores(auto_scores, "SAC (v2, auto-alpha) on Pendulum")
print(f"final learned alpha: {auto_agent.alpha:.4f}")

In [ ]:
display(render_gif("Pendulum-v1", auto_agent, "video/sac_v2_pendulum.gif", steps=200))

## Short comparison — SAC on Walker2d-v5 (vs Day 5's DDPG/TD3)

The exact same agent handles a much harder task. **Walker2d** is the MuJoCo locomotion
env from Day 5, where DDPG was brittle and TD3 was steadier. SAC learns it too — with
*no* injected exploration noise — thanks to the entropy bonus. We use the auto-$\alpha$
agent because it sidesteps the reward-scale sensitivity the lecture flags for fixed-$\alpha$
SAC v1.

> **Note:** this is CPU-slow and deliberately short — the return should trend upward but
> won't reach a polished gait here. Bump `walker_steps` to 300k+ (ideally on GPU) for a
> real walk, exactly as on Day 5.

In [ ]:
walker_env = gym.make("Walker2d-v5", render_mode="rgb_array")
walker_args = dict(args, start_steps=10_000, alpha_lr=3e-4)
walker_steps = 30_000       # short demo; 300k+ for a real gait

walker_buffer = ExperienceBuffer(walker_args["replay_size"])
walker_agent = SACAutoAlpha(walker_env, walker_buffer, walker_args)

walker_scores = train(walker_agent, walker_env, walker_steps, walker_args)
plot_scores(walker_scores, "SAC (auto-alpha) on Walker2d — short CPU demo")

In [ ]:
display(render_gif("Walker2d-v5", walker_agent, "video/sac_walker.gif", steps=500))